# Chunking / encoding read benchmark

Compares read latency for the same QA output across two icechunk branches:
the `chunking_and_codecs` layout (small space-major chunks + shards + Blosc) and
the previous time-major layout.

Both branches live in the **same** icechunk repository -- the store path does not
encode the branch -- so only `branch` changes between the two reads.

Run on an in-region VM (`us-west-2`). Cross-region latency swamps the effect.

In [ ]:
import statistics
import time

import icechunk
import pandas as pd

from saidownscale.bcsd_config import BCSDConfig, PipelineOptions
from saidownscale.cache import ArtifactCache
from saidownscale.utils import open_icechunk

# --- params: match the QA config that produced the run ---
GCM = "CESM2-WACCM"
VARIABLE = "tas"
MEMBER = "001"
SCENARIO = "G6-1.5K"
SUBSET_BOUNDS = (-38.0, -19.0, 13.0, 36.0)

OUTPUT_DIR = "s3://carbonplan-srm/scratch/output"
SCRATCH_DIR = "s3://carbonplan-srm/scratch/cache"
ENVIRONMENT = "qa"

BRANCH_NEW = "chunking_encoding_a2"
BRANCH_OLD = "main"  # set from the branch listing in the next cell

N_TRIALS = 3

In [ ]:
config = BCSDConfig(
    gcm=GCM,
    variable=VARIABLE,
    ensemble_member=MEMBER,
    scenario=SCENARIO,
    subset_bounds=SUBSET_BOUNDS,
)
options = PipelineOptions(
    scratch_dir=SCRATCH_DIR,
    output_dir=OUTPUT_DIR,
    environment=ENVIRONMENT,
)
loc = ArtifactCache.from_config(config, options).scenario_loc
STORE, GROUP = loc.store_path, loc.group

bucket, prefix = STORE.replace("s3://", "").rstrip("/").split("/", 1)
repo = icechunk.Repository.open(icechunk.s3_storage(bucket=bucket, prefix=prefix))

print(f"store: {STORE}")
print(f"group: {GROUP}")
print(f"branches: {sorted(repo.list_branches())}")

In [ ]:
# On-disk layout on each branch. Confirms the two runs actually differ.
for label, branch in [("new", BRANCH_NEW), ("old", BRANCH_OLD)]:
    ds = open_icechunk(STORE, group=GROUP, branch=branch)
    da = ds[VARIABLE]
    enc = da.encoding
    print(f"[{label}] branch={branch}")
    print(f"  shape       {da.shape}  ({da.nbytes / 1e9:.2f} GB uncompressed)")
    print(f"  chunks      {enc.get('chunks')}")
    print(f"  shards      {enc.get('shards')}")
    print(f"  compressors {enc.get('compressors')}")
    print(f"  dtype       {da.dtype}")

In [ ]:
def bench(branch: str, selector, n: int = N_TRIALS) -> list[float]:
    """Time `selector(ds).load()`, reopening the store each trial.

    Reopening drops the xarray/zarr object cache. OS page cache and any obstore
    cache still persist, so treat trial 1 as the cold number and later trials as warm.
    """
    times = []
    for _ in range(n):
        ds = open_icechunk(STORE, group=GROUP, branch=branch)
        t0 = time.perf_counter()
        selector(ds[VARIABLE]).load()
        times.append(time.perf_counter() - t0)
    return times


_probe = open_icechunk(STORE, group=GROUP, branch=BRANCH_NEW)[VARIABLE]
mid_lat, mid_lon = _probe.sizes["lat"] // 2, _probe.sizes["lon"] // 2
mid_time = _probe.sizes["time"] // 2

ACCESS_PATTERNS = {
    # Time-major: one grid cell, full record. Favors long time chunks.
    "timeseries_point": lambda da: da.isel(lat=mid_lat, lon=mid_lon),
    # Space-major: one timestep, full domain. Favors wide lat/lon chunks.
    "map_single_time": lambda da: da.isel(time=mid_time),
    # One year, full domain. Typical QA/plotting slice.
    "map_one_year": lambda da: da.isel(time=slice(mid_time, mid_time + 365)),
    # Small spatial box, full record. Regional timeseries.
    "timeseries_box": lambda da: da.isel(
        lat=slice(mid_lat - 2, mid_lat + 2), lon=slice(mid_lon - 2, mid_lon + 2)
    ),
}

In [ ]:
rows = []
for name, selector in ACCESS_PATTERNS.items():
    for label, branch in [("new", BRANCH_NEW), ("old", BRANCH_OLD)]:
        t = bench(branch, selector)
        rows.append(
            {
                "pattern": name,
                "layout": label,
                "cold_s": round(t[0], 3),
                "best_s": round(min(t), 3),
                "median_s": round(statistics.median(t), 3),
            }
        )
        print(f"{name:20s} {label:3s} {[round(x, 3) for x in t]}")

results = pd.DataFrame(rows)
results

In [ ]:
pivot = results.pivot(index="pattern", columns="layout", values="median_s")
pivot["speedup_new_vs_old"] = (pivot["old"] / pivot["new"]).round(2)
print(pivot)

ax = pivot[["old", "new"]].plot.barh(figsize=(7, 3.5))
ax.set_xlabel("median load time (s)")
ax.set_ylabel("")
ax.set_title(f"{GCM} {VARIABLE} {MEMBER} {SCENARIO}")

## What to expect

For the South Africa QA box (~76 x 92 fine cells, ~18k timesteps over 2035-2084),
bytes fetched per pattern:

| pattern | old `(8000, 8, 16)` | new `(365, 36, 72)` |
|---|---|---|
| `timeseries_point` | ~3 chunks, ~12 MB | ~51 chunks, ~180 MB |
| `map_single_time` | ~60 chunks, ~240 MB | ~6 chunks, ~21 MB |

So the new layout should lose on point timeseries and win on maps. If the
measured numbers do not show that shape, the benchmark is measuring something
else -- most likely network variance or a warm cache. Re-run with `N_TRIALS`
raised and compare `cold_s` separately from `median_s`.

Sharding means zarr issues byte-range requests inside a shard rather than
fetching whole shards, so request *count* matters as much as bytes. On a
high-latency link the chunk count dominates; in-region, bytes dominate.